# Utilization Prediction

## a) Business Case

During the descriptives task, we defined three KPIs:

1) Utilization rate
2) Energy delivered per hour
3) Rate of registered user sessions

The business case behind these KPIs is defined as:

<br><br>
<center>
  <b style="font-size: 1.3em;">
    The prediction model enables the site operator to offer dynamic pricing to registered users to maximize revenue and energy throughput while smoothing demand.
  </b>
</center>
<br><br>

The model predicts the hourly utilization of a given site. This information can be used to offer dynamic pricing models: when utilization and energy delivered are low, prices can be lowered to encourage usage. Notifications can be sent to registered users, e.g. through an app, to increase engagement, improve site utilization and maximize energy throughput.


## b) Prediction Model

In [3]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import seaborn as sns
import holidays

# Load prepared dataframes from pickle files
data_path = "../data_processed/"

with open(data_path + "df_prepared.pkl", "rb") as f:
    df = pickle.load(f)

with open(data_path + "merged_df_prepared.pkl", "rb") as f:
    merged_df = pickle.load(f)

with open(data_path + "relevant_weather_data_prepared.pkl", "rb") as f:
    relevant_weather_data = pickle.load(f)

print("DataFrames loaded successfully:")
print(f"  df shape: {df.shape}")
print(f"  merged_df shape: {merged_df.shape}")
print(f"  relevant_weather_data shape: {relevant_weather_data.shape}")

DataFrames loaded successfully:
  df shape: (65006, 21)
  merged_df shape: (65006, 27)
  relevant_weather_data shape: (26187, 9)


We choose site 1 for this task

In [4]:
print(merged_df.siteID.unique())
print(merged_df.stationID.dtype)
df_site_1 = merged_df[merged_df['siteID'] == '1']
df_site_1 = df_site_1.sort_values(by='connectionTime', ascending=True)
df_site_1.head()


['1' '2']
object


,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,userID,...,session_duration,charging_duration,isRegisteredUser,WhPerMile,kWhRequested,milesRequested,minutesAvailable,modifiedAt,paymentRequired,requestedDeparture
11686,5c36621bf9af8b4639a8e0b4,2018-09-05 04:04:13-07:00,2018-09-05 12:09:35-07:00,2018-09-05 12:09:35-07:00,9.583,1_1_179_800_2018-09-05 11:04:12.876087,1,ag-3f32,1-1-179-800,-1,...,8.089444,8.089444,False,NaN,NaN,NaN,NaN,NaT,NaN,NaT
8985,5c36621bf9af8b4639a8e0b5,2018-09-05 04:08:09-07:00,2018-09-05 07:09:02-07:00,2018-09-05 07:09:02-07:00,7.114,1_1_179_794_2018-09-05 11:08:08.945820,1,ag-3f20,1-1-179-794,333.0,...,3.014722,3.014722,True,400.0,8.0,20.0,421.0,2018-09-05 06:32:58-07:00,True,2018-09-05 11:09:09-07:00
10020,5c36621bf9af8b4639a8e0b6,2018-09-05 05:35:14-07:00,2018-09-05 17:30:12-07:00,2018-09-05 17:30:12-07:00,11.774,1_1_179_797_2018-09-05 12:35:14.070250,1,ag-3f23,1-1-179-797,371.0,...,11.916111,11.916111,True,600.0,24.0,40.0,643.0,2018-09-05 06:16:35-07:00,True,2018-09-05 16:18:14-07:00
5685,5c36621bf9af8b4639a8e0b7,2018-09-05 05:51:31-07:00,2018-09-05 15:32:58-07:00,2018-09-05 15:32:58-07:00,6.280,1_1_179_781_2018-09-05 12:51:31.050539,1,ag-3f31,1-1-179-781,405.0,...,9.690833,9.690833,True,500.0,10.0,20.0,576.0,2018-09-05 14:10:15-07:00,True,2018-09-05 15:27:31-07:00
6810,5c36621bf9af8b4639a8e0b8,2018-09-05 06:08:28-07:00,2018-09-05 16:32:52-07:00,2018-09-05 16:32:52-07:00,7.022,1_1_179_787_2018-09-05 13:08:27.901538,1,ag-3f16,1-1-179-787,368.0,...,10.406667,10.406667,True,400.0,8.0,20.0,608.0,2018-09-05 06:36:15-07:00,True,2018-09-05 16:16:28-07:00


In [5]:
mask = df['charging_duration'] == df['session_duration']
anzahl = df[mask].shape[0]
print(f"Anzahl der Sitzungen, bei denen die Ladedauer gleich der Sitzungsdauer ist: {anzahl}")

Anzahl der Sitzungen, bei denen die Ladedauer gleich der Sitzungsdauer ist: 8479


In [6]:
print(df_site_1['stationID'].nunique())

52


## Calculate Utilization for all hour where there was a session

In [7]:
def make_site1_hourly_utilization(df,
                                  conn_col='connectionTime',
                                  disc_col='disconnectTime',
                                  station_col='stationID'):
    df = df.copy()
    df[conn_col] = pd.to_datetime(df[conn_col], errors='coerce')
    df[disc_col] = pd.to_datetime(df[disc_col], errors='coerce')

    n_stations = df[station_col].nunique()

    def row_hour_contrib(row):
        start = row[conn_col]
        end = row[disc_col]
        if pd.isna(start) or pd.isna(end) or end <= start:
            return []
        hours = pd.date_range(
            start=start.floor('H'),
            end=(end - pd.Timedelta(seconds=1)).floor('H'),
            freq='H'
        )
        out = []
        for h in hours:
            h_start = max(start, h)
            h_end = min(end, h + pd.Timedelta(hours=1))
            minutes = (h_end - h_start).total_seconds() / 60.0
            if minutes > 0:
                out.append((h, minutes))
        return out

    contrib = df.apply(row_hour_contrib, axis=1)
    tmp = pd.DataFrame(
        [{'datetime': h, 'minutes': m}
         for lst in contrib for (h, m) in lst]
    )

    if tmp.empty:
        hourly = pd.DataFrame(
            columns=['datetime', 'hourly_utilization_minutes',
                     'hourly_utilization']
        )
        hourly['datetime'] = pd.to_datetime(hourly['datetime'])
        return hourly.set_index('datetime')

    # 1) Aggregation: eine Zeile pro Stunde
    hourly = (
        tmp.groupby('datetime')['minutes']
        .sum()
        .rename('hourly_utilization_minutes')
        .to_frame()
    )

    hourly['hourly_utilization'] = (
        hourly['hourly_utilization_minutes'] / (n_stations * 60.0)
    ).clip(0, 1)



    return hourly



In [8]:
# Compute hourly utilization for site 1 using connectionTime & disconnectTime
df_site_1_util = make_site1_hourly_utilization(df_site_1, 'connectionTime', 'disconnectTime')
df_site_1 = df_site_1.sort_values(by='connectionTime', ascending=True)


C:\Users\Daniel\AppData\Local\Temp\ipykernel_11416\3120264220.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  start=start.floor('H'),
C:\Users\Daniel\AppData\Local\Temp\ipykernel_11416\3120264220.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  end=(end - pd.Timedelta(seconds=1)).floor('H'),
C:\Users\Daniel\AppData\Local\Temp\ipykernel_11416\3120264220.py:16: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(


In [9]:
df_site_1_util.head()


,hourly_utilization_minutes,hourly_utilization
datetime,,
2018-09-05 04:00:00-07:00,107.633333,0.034498
2018-09-05 05:00:00-07:00,153.250000,0.049119
2018-09-05 06:00:00-07:00,469.800000,0.150577
2018-09-05 07:00:00-07:00,874.300000,0.280224
2018-09-05 08:00:00-07:00,900.000000,0.288462


## Holidays

**pip install holidays runnen**

In [10]:
US_holidays = holidays.US(state='CA')

df_site_1_util['is_holiday'] = df_site_1_util.index.to_series().apply(lambda dt: dt in US_holidays)
df_site_1_util['holiday_name'] = df_site_1_util.index.to_series().apply(lambda dt: US_holidays.get(dt) if dt in US_holidays else '')

df_site_1_util.head()



,hourly_utilization_minutes,hourly_utilization,is_holiday,holiday_name
datetime,,,,
2018-09-05 04:00:00-07:00,107.633333,0.034498,False,
2018-09-05 05:00:00-07:00,153.250000,0.049119,False,
2018-09-05 06:00:00-07:00,469.800000,0.150577,False,
2018-09-05 07:00:00-07:00,874.300000,0.280224,False,
2018-09-05 08:00:00-07:00,900.000000,0.288462,False,


## Create hourly data of sessions

In [11]:
def make_site1_hourly_features(df,
                               conn_col='connectionTime',
                               disc_col='disconnectTime',
                               station_col='stationID'):
    df = df.copy()
    df[conn_col] = pd.to_datetime(df[conn_col], errors='coerce')
    df[disc_col] = pd.to_datetime(df[disc_col], errors='coerce')

    n_stations = df[station_col].nunique()

    def row_hour_contrib(row):
        start = row[conn_col]
        end = row[disc_col]
        if pd.isna(start) or pd.isna(end) or end <= start:
            return []

        hours = pd.date_range(
            start=start.floor('H'),
            end=(end - pd.Timedelta(seconds=1)).floor('H'),
            freq='H'
        )

        out = []
        for h in hours:
            h_start = max(start, h)
            h_end = min(end, h + pd.Timedelta(hours=1))
            minutes = (h_end - h_start).total_seconds() / 60.0
            if minutes > 0:
                out.append({
                    'datetime': h,
                    'minutes': minutes,
                    'kWhDelivered': row['kWhDelivered'],
                    'userID': row['userID'],
                    'siteID': row[station_col],
                    'is_registered': row['isRegisteredUser'],
                    'session_duration': row['session_duration'],
                    'charging_duration': row['charging_duration']

                })
        return out

    contrib = df.apply(row_hour_contrib, axis=1)
    tmp = pd.DataFrame([item for lst in contrib for item in lst])

    if tmp.empty:
        return pd.DataFrame(columns=[
            'datetime', 'hourly_utilization_minutes', 'hourly_utilization'
        ]).set_index('datetime')

    # Stundenaggregation für Nutzung + zusätzliche Features
    hourly = (
        tmp.groupby('datetime')
            .agg(
                hourly_utilization_minutes=('minutes', 'sum'),
                energy_kwh_hour=('kWhDelivered', 'sum'),
                avg_kwh_per_session=('kWhDelivered', 'mean'),
                avg_session_duration=('session_duration', 'mean'),
                avg_charging_duration=('charging_duration', 'mean'),
                n_sessions=('userID', 'nunique'),
                n_registered_sessions=('is_registered', 'sum')
            )
    )

    hourly['hourly_utilization'] = (
        hourly['hourly_utilization_minutes'] / (n_stations * 60.0)
    ).clip(0, 1)

    return hourly

    

In [12]:
df_site_1_features = make_site1_hourly_features(df_site_1, 'connectionTime', 'disconnectTime')
df_site_1_features.head()

C:\Users\Daniel\AppData\Local\Temp\ipykernel_11416\670990831.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  start=start.floor('H'),
C:\Users\Daniel\AppData\Local\Temp\ipykernel_11416\670990831.py:19: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  end=(end - pd.Timedelta(seconds=1)).floor('H'),
C:\Users\Daniel\AppData\Local\Temp\ipykernel_11416\670990831.py:17: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hours = pd.date_range(


,hourly_utilization_minutes,energy_kwh_hour,avg_kwh_per_session,avg_session_duration,avg_charging_duration,n_sessions,n_registered_sessions,hourly_utilization
datetime,,,,,,,,
2018-09-05 04:00:00-07:00,107.633333,16.697,8.348500,5.552083,5.552083,2,1,0.034498
2018-09-05 05:00:00-07:00,153.250000,34.751,8.687750,8.177778,8.177778,4,3,0.049119
2018-09-05 06:00:00-07:00,469.800000,110.399,10.036273,8.564394,8.564394,10,9,0.150577
2018-09-05 07:00:00-07:00,874.300000,158.350,9.896875,8.464219,8.464219,10,9,0.280224
2018-09-05 08:00:00-07:00,900.000000,151.236,10.082400,8.827519,8.827519,9,8,0.288462


We add a feature for holidays as this  influence the utilization due to the fact that employees do not work at this and university is also closed there. Additionally we add the already know time features from the session dataset.

In [13]:
US_holidays = holidays.US(state='CA')

# Ensure index is datetime
df_site_1_features.index = pd.to_datetime(df_site_1_features.index)

df_site_1_features['is_holiday'] = df_site_1_features.index.to_series().apply(lambda dt: dt in US_holidays)
df_site_1_features['holiday_name'] = df_site_1_features.index.to_series().apply(lambda dt: US_holidays.get(dt) if dt in US_holidays else '')

#Add time features based on the index
df_site_1_features['year'] = df_site_1_features.index.year
df_site_1_features['month'] = df_site_1_features.index.month
df_site_1_features['day'] = df_site_1_features.index.day
df_site_1_features['hour'] = df_site_1_features.index.hour
df_site_1_features['dayofweek'] = df_site_1_features.index.dayofweek
df_site_1_features['is_weekend'] = df_site_1_features['dayofweek'].apply(lambda x: 1 if x >= 5 else 0)

def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'
    
df_site_1_features['season'] = df_site_1_features['month'].apply(get_season)

In [14]:
df_site_1_features.head()

,hourly_utilization_minutes,energy_kwh_hour,avg_kwh_per_session,avg_session_duration,avg_charging_duration,n_sessions,n_registered_sessions,hourly_utilization,is_holiday,holiday_name,year,month,day,hour,dayofweek,is_weekend,season
datetime,,,,,,,,,,,,,,,,,
2018-09-05 04:00:00-07:00,107.633333,16.697,8.348500,5.552083,5.552083,2,1,0.034498,False,,2018,9,5,4,2,0,Fall
2018-09-05 05:00:00-07:00,153.250000,34.751,8.687750,8.177778,8.177778,4,3,0.049119,False,,2018,9,5,5,2,0,Fall
2018-09-05 06:00:00-07:00,469.800000,110.399,10.036273,8.564394,8.564394,10,9,0.150577,False,,2018,9,5,6,2,0,Fall
2018-09-05 07:00:00-07:00,874.300000,158.350,9.896875,8.464219,8.464219,10,9,0.280224,False,,2018,9,5,7,2,0,Fall
2018-09-05 08:00:00-07:00,900.000000,151.236,10.082400,8.827519,8.827519,9,8,0.288462,False,,2018,9,5,8,2,0,Fall


In [15]:
df_site_1_features.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 16123 entries, 2018-09-05 04:00:00-07:00 to 2021-09-14 07:00:00-07:00
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   hourly_utilization_minutes  16123 non-null  float64
 1   energy_kwh_hour             16123 non-null  float64
 2   avg_kwh_per_session         16123 non-null  float64
 3   avg_session_duration        16123 non-null  float64
 4   avg_charging_duration       16123 non-null  float64
 5   n_sessions                  16123 non-null  int64  
 6   n_registered_sessions       16123 non-null  int64  
 7   hourly_utilization          16123 non-null  float64
 8   is_holiday                  16123 non-null  bool   
 9   holiday_name                16123 non-null  object 
 10  year                        16123 non-null  int32  
 11  month                       16123 non-null  int32  
 12  day                         16123 non-nul

In [ ]:
df_site_1_features['util_last_hour_flag'] = (df_site_1_features['hourly_utilization'].shift(1) > 0).astype(int)

KeyError: 'hourly_utilization'

## c) Concrete Example of Application

TODO